# Lesson 02 - Eksploracja Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** to zunifikowany framework do tworzenia agentów AI. Zapewnia czystą, kompozycyjną architekturę z czterema głównymi blokami budulcowymi:

- **Client** – łączy się z punktem końcowym modelu AI i obsługuje komunikację
- **Agent** – opakowuje klienta z instrukcjami i definicjami narzędzi
- **Tools** – rozszerzają możliwości agenta o niestandardowe funkcje, które model może wywoływać
- **Session** – utrzymuje historię rozmowy dla interakcji wielokrotnych

W tej lekcji zbudujemy **agenta do rezerwacji podróży**, który sprawdza dostępność miejsc docelowych, wykorzystując te koncepcje.


## Instalacja


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Zrozumienie architektury Agent Framework

Microsoft Agent Framework opiera się na architekturze warstwowej:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – `AzureAIProjectAgentProvider` łącze się z deploymentem Azure OpenAI. Obsługuje uwierzytelnianie, formatowanie zapytań oraz analizę odpowiedzi.
2. **Agent** – Tworzony z klienta za pomocą `provider.create_agent()`, agent łączy dostęp do modelu z instrukcjami (system prompt) oraz narzędziami.
3. **Tools** – Funkcje Pythona dekorowane za pomocą `@tool`, które agent może wywoływać, aby wykonać akcje lub pobrać dane.
4. **Session** – Obiekt `AgentSession` (tworzony za pomocą `agent.create_session()`), który przechowuje historię rozmowy, umożliwiając dialog wieloobrotowy, gdzie agent pamięta wcześniejszy kontekst.

Zbudujmy każdą warstwę krok po kroku.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Dodawanie narzędzi za pomocą dekoratora @tool

Narzędzia pozwalają agentom wykonywać działania wykraczające poza generowanie tekstu. Dekorator `@tool` zamienia zwykłą funkcję Pythona w coś, co agent może wywołać.

Kluczowe punkty:
- Używaj `Annotated[type, "description"]`, aby model rozumiał każdy parametr.
- Docstring staje się opisem narzędzia, który widzi model.
- `approval_mode="never_require"` oznacza, że narzędzie działa automatycznie bez potwierdzenia użytkownika.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Tworzenie agenta z narzędziami

Teraz łączymy klienta, instrukcje i narzędzia w agenta. `instructions` działają jako prompt systemowy — definiują osobowość i zachowanie agenta.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Konwersacje wieloetapowe z sesjami

`AgentSession` (tworzony za pomocą `agent.create_session()`) śledzi wszystkie wiadomości w rozmowie. Przekazując tę samą sesję do każdego wywołania `agent.run()`, agent ma dostęp do pełnej historii rozmowy i może odwoływać się do wcześniejszych wiadomości.

Przekazujemy `tools=[check_destination_availability]`, aby agent mógł wywoływać nasz sprawdzacz dostępności podczas każdej tury.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Podsumowanie

W tej lekcji poznano cztery filary Microsoft Agent Framework:

| Pojęcie | Czego się nauczyłeś |
|---------|---------------------|
| **Klient** | `AzureAIProjectAgentProvider` łączy się z Azure OpenAI za pomocą uwierzytelniania opartego na poświadczeniach |
| **Agent** | `provider.create_agent()` łączy połączenie modelu z instrukcjami i nazwą |
| **Narzędzia** | Dekorator `@tool` udostępnia funkcje Pythona, które agent może wywoływać |
| **Sesja** | `agent.create_session()` utrzymuje historię rozmowy przez wiele tur |

Te elementy składowe łączą się, tworząc agentów zdolnych do prowadzenia naturalnych rozmów, wywoływania zewnętrznych funkcji oraz utrzymywania kontekstu — co stanowi podstawę bardziej zaawansowanych wzorców agentskich w kolejnych lekcjach.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:  
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mimo że dążymy do dokładności, prosimy mieć na uwadze, że tłumaczenia automatyczne mogą zawierać błędy lub nieścisłości. Oryginalny dokument w języku źródłowym należy traktować jako autorytatywne źródło. W przypadku informacji o kluczowym znaczeniu zaleca się skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z korzystania z tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
